# EMA 6938 — Data Science for Materials
## Week 2 Notebook: Data Representation, Materials Databases, & Experimental Design

**Chapters:** Sandfeld Ch. 3–4  
**Parts A–C:** Completed during Day 1 lab (75 min)  
**Parts D–E:** Take-home — due **Sunday 11:59 PM** via Canvas

---

### Files required
Place these in the same directory as this notebook (available on course GitHub under `week02/`):
```
week02/
├── week2_data_databases.ipynb   ← this file
├── data/
│   ├── week2_tg_polymer.csv
│   └── week2_dirty_mp_subset.csv
└── structures/
    ├── BaTiO3.cif
    └── TiO2_rutile.poscar
```

### Environment check
Run the cell below before anything else. If any import fails, install the missing package and restart the kernel.

---

In [ ]:
# Cell 0 — Environment check
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymatgen
from pymatgen.core import Structure, Composition

try:
    import jarvis
    jarvis_ok = True
except ImportError:
    jarvis_ok = False

print(f"Python:     {sys.version.split()[0]}")
print(f"NumPy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"pymatgen:   {pymatgen.core.__version__}")
print(f"jarvis-tools installed: {jarvis_ok}")
if not jarvis_ok:
    print("\n  → Install: pip install jarvis-tools  (then restart kernel)")
print("\n✓ Environment ready.")

---
## Part A — Materials Data Structures
**In-class · Day 1 · Segment 1 (0–15 min)**

You will use three data containers all semester. This part shows you the same BaTiO₃ information in each container — so you can see what each one is good for.

| Container | Best for | Lives inside |
|---|---|---|
| `numpy.ndarray` | Matrix math, lattice operations | pandas DataFrames, Structure |
| `pandas.DataFrame` | Multi-property tabular datasets | Your analysis code |
| `pymatgen.Structure` | Crystal geometry, atomic positions | Fetched from databases |

**The hierarchy:** Structure objects are often converted to DataFrames for ML. Arrays live inside DataFrames. Understanding the conversions is half the job.

In [ ]:
# Cell A1 — NumPy: lattice parameters as a matrix
# INSTRUCTOR DEMO — follow along

import numpy as np

# BaTiO3 tetragonal lattice parameters (Angstrom)
a, b, c = 3.994, 3.994, 4.038

# The 3×3 lattice matrix — rows are lattice vectors
lattice = np.array([[a, 0, 0],
                    [0, b, 0],
                    [0, 0, c]])

print("Lattice matrix (Å):")
print(lattice)
print()
print(f"Volume (Å³): {np.linalg.det(lattice):.4f}")
print(f"dtype:       {lattice.dtype}")
print(f"shape:       {lattice.shape}")
print()
print("Why a matrix? Lattice vectors define the unit cell.")
print("Volume = det(lattice) — a one-liner with NumPy.")

In [ ]:
# Cell A2 — pandas: same information in a DataFrame
# INSTRUCTOR DEMO

import pandas as pd

df_struct = pd.DataFrame({
    'material': ['BaTiO3'],
    'crystal_system': ['tetragonal'],
    'a_Ang': [a],
    'b_Ang': [b],
    'c_Ang': [c],
    'c_over_a': [c / a],
    'volume_Ang3': [np.linalg.det(lattice)],
})

print(df_struct.to_string(index=False))
print()
print("Why a DataFrame? When you have 10,000 materials, each is a row.")
print("c/a ratio encodes the tetragonal distortion → ferroelectricity.")

In [ ]:
# Cell A3 — pandas task: add SrTiO3
# YOUR CODE HERE

# SrTiO3 is cubic: a = b = c = 3.905 Å, all angles = 90°
# Add a new row for SrTiO3 to df_struct
# Hint: use pd.concat([df_struct, new_row], ignore_index=True)

# YOUR CODE HERE


# After adding: print the updated DataFrame
# and answer: how does the c/a ratio of BaTiO3 vs SrTiO3 connect to ferroelectricity?


In [ ]:
# Cell A4 — Pymatgen Structure object
# INSTRUCTOR DEMO

from pymatgen.core import Structure

# Load from the provided CIF file
struct = Structure.from_file('structures/BaTiO3.cif')

print(struct)
print()
print(f"Formula:      {struct.formula}")
print(f"Space group:  {struct.get_space_group_info()}")
print(f"Num sites:    {len(struct)}")
print(f"Volume (Å³):  {struct.volume:.4f}")
print()
print("Lattice from Structure object:")
print(f"  a={struct.lattice.a:.4f} Å, b={struct.lattice.b:.4f} Å, c={struct.lattice.c:.4f} Å")

In [ ]:
# Cell A5 — Task: find the Ti site and print its fractional coordinates
# YOUR CODE HERE

# 1. Loop over struct and find the site(s) where species_string == 'Ti'
# 2. Print the fractional coordinates of the Ti site
# 3. Compare to (0.5, 0.5, 0.5) — what does the offset tell you?

# YOUR CODE HERE


In [ ]:
# Cell A6 — Converting between containers
# INSTRUCTOR DEMO

# Structure → NumPy (the lattice matrix)
lat_array = struct.lattice.matrix   # already a numpy array
print("Lattice matrix from Structure (NumPy):")
print(lat_array)
print(f"\nVolume from numpy: {np.linalg.det(lat_array):.4f} Å³")
print(f"Volume from struct: {struct.volume:.4f} Å³")
print()

# Structure → pandas row
row = {
    'formula': struct.formula,
    'sg_number': struct.get_space_group_info()[1],
    'a': struct.lattice.a,
    'b': struct.lattice.b,
    'c': struct.lattice.c,
    'volume': struct.volume,
    'n_sites': len(struct),
}
df_from_struct = pd.DataFrame([row])
print("Structure → DataFrame row:")
print(df_from_struct.to_string(index=False))

**A6 Reflection** *(answer in this markdown cell)*

In one sentence each:
- When would you use a NumPy array instead of a Structure object?
- When would you use a pandas DataFrame instead of either?

*Your answer here:*

---
## Part B — Materials Databases
**In-class · Day 1 · Segment 2 (15–40 min)**

No single database is the ground truth. The choice of database is a scientific decision — it encodes assumptions about level of theory, convergence criteria, and what properties matter. **Always cite the database and version in your methods section.**

| Database | Entries | Functional | Best for |
|---|---|---|---|
| Materials Project | ~154k | GGA-PBE | Inorganic crystals, broad coverage |
| JARVIS-DFT | ~80k | PBE + optB88vdW | 2D materials, mechanical props |
| AFLOW | ~3.5M | Various | Alloys, high-throughput screening |
| ICSD | — | Experimental | Real structure determination data |

In [ ]:
import subprocess
import sys

try:
    import jarvis
except ImportError:
    print("Installing jarvis-tools...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "jarvis-tools", "-q"])
    print("✓ jarvis-tools installed. Restart kernel and re-run cells.")

In [ ]:
# Cell B1 — Fetch from JARVIS-DFT
# INSTRUCTOR DEMO — first load takes ~2 min; subsequent calls are instant

from jarvis.db.figshare import data as jdata

print("Loading JARVIS DFT-3D database (~80k entries)...")
print("First load: ~2 min (cached after that). Use this time for questions.")

jd = {e['jid']: e for e in jdata('dft_3d')}
print(f"\nLoaded {len(jd):,} JARVIS entries.")

# TiO2 rutile in JARVIS
entry = jd['JVASP-1177']

print(f"\nJARVIS TiO₂ rutile (JVASP-1177):")
print(f"  Formula:            {entry['formula']}")
print(f"  Bandgap (optB88vdW): {entry['optb88vdw_bandgap']:.4f} eV")
print(f"  Formation energy:    {entry['formation_energy_peratom']:.4f} eV/atom")
print(f"  Space group:         {entry['spg_symbol']}")

In [ ]:
# Cell B2 — Same compound from Materials Project
# INSTRUCTOR DEMO

from mp_api.client import MPRester
from dotenv import load_dotenv
import os

load_dotenv()  # loads MP_API_KEY from .env file
API_KEY = os.getenv('MP_API_KEY', 'YOUR_API_KEY_HERE')

with MPRester(API_KEY) as mpr:
    mp_entry = mpr.materials.summary.get_data_by_id(
        'mp-2657',
        fields=['material_id', 'formula_pretty', 'band_gap',
                'formation_energy_per_atom', 'energy_above_hull']
    )

print(f"MP TiO₂ rutile (mp-2657):")
print(f"  Formula:          {mp_entry.formula_pretty}")
print(f"  Bandgap (GGA-PBE): {mp_entry.band_gap:.4f} eV")
print(f"  Formation energy:  {mp_entry.formation_energy_per_atom:.4f} eV/atom")
print(f"  E above hull:      {mp_entry.energy_above_hull:.4f} eV/atom")

In [ ]:
# Cell B3 — Side-by-side comparison
# INSTRUCTOR DEMO

jarvis_gap = entry['optb88vdw_bandgap']
mp_gap     = mp_entry.band_gap

print("TiO₂ rutile — database comparison:")
print(f"  JARVIS (optB88vdW): {jarvis_gap:.4f} eV")
print(f"  MP     (GGA-PBE):   {mp_gap:.4f} eV")
print(f"  |Δ|:                {abs(jarvis_gap - mp_gap):.4f} eV")
print()
print("Experimental TiO₂ bandgap: ~3.0 eV (indirect)")
print("Both DFT functionals underestimate — but by different amounts.")
print("optB88vdW adds van der Waals corrections → different electron density → different gap.")

In [ ]:
# Cell B4 — Task: fetch a second compound of your choice from JARVIS
# YOUR CODE HERE

# 1. Browse JARVIS at https://jarvis.nist.gov or search jd.keys()
#    to find a JID for a material related to your research.
#    Hint: search by formula: [v for k,v in jd.items() if v['formula']=='ZnO']

# 2. Fetch its bandgap and formation energy from JARVIS

# 3. Fetch the same compound from Materials Project (find the mp-id at materialsproject.org)

# 4. Print a comparison table: property | JARVIS value | MP value | |difference|

# YOUR CODE HERE


**B4 Reflection** *(answer in this markdown cell)*

1. What material did you choose and why?
2. Which database gave a larger bandgap, and does this agree with the pattern seen for TiO₂?
3. In one sentence: if you were training an ML model to predict bandgap for your material class, which database would you prefer and why?

*Your answer here:*

In [ ]:
# Cell B5 — Data quality: what 'unconverged' means
# INSTRUCTOR DEMO

# In JARVIS, check the 'magmom_oszicar' field — it records magnetic moment from
# the OSZICAR file. If EDIFF is not met, the entry may be flagged.
# In MP, use energy_above_hull > 0.1 eV/atom as a rough stability filter.

print("Data quality indicators:")
print()
print("JARVIS — useful quality fields:")
quality_fields = ['atoms_icsd', 'magmom_oszicar', 'gap_opt', 'encut']
for f in quality_fields:
    val = entry.get(f, 'field not present')
    print(f"  {f}: {val}")

print()
print("MP — energy above hull as stability proxy:")
print(f"  mp-2657 E_hull = {mp_entry.energy_above_hull:.4f} eV/atom")
print("  Rule of thumb: E_hull > 0.1 eV/atom → likely metastable or unconverged")
print()
print("Implication for ML: unconverged entries add noise to training sets.")
print("Always apply a quality filter before training — we do this in Part C.")

---
## Part C — Tg Polymer Dataset & Experimental Design Audit
**In-class · Day 1 · Segments 3–4 + Lab (40–150 min)**

The dataset in `week2_tg_polymer.csv` contains 312 entries with **deliberate quality issues** for you to find:
- 3 duplicate entries with conflicting Tg values
- 11 missing molecular weight values
- 4 samples with implausibly low Tg (<150 K) — likely unit errors (°C entered as K)

Finding and fixing these is the core experimental design exercise — it mirrors what you will do with every real dataset you use for ML.

In [ ]:
# Cell C1 — Load and inspect the dataset
# INSTRUCTOR DEMO

import pandas as pd
import matplotlib.pyplot as plt

# Windows users: use encoding='utf-8-sig' if you see a BOM character in column headers
df_tg = pd.read_csv('data/week2_tg_polymer.csv')

print(f"Shape: {df_tg.shape[0]} rows × {df_tg.shape[1]} columns")
print()
print("First 10 rows:")
print(df_tg.head(10).to_string())
print()
print("Summary statistics:")
print(df_tg.describe())
print()
print("Missing values per column:")
print(df_tg.isnull().sum())
print()
print("Column dtypes:")
print(df_tg.dtypes)

In [ ]:
# Cell C2 — First visualization: Tg vs molecular weight
# INSTRUCTOR DEMO

fig, ax = plt.subplots(figsize=(9, 6))

# Colour by monomer type
monomer_codes = df_tg['monomer_type'].astype('category').cat.codes
monomer_names = df_tg['monomer_type'].astype('category').cat.categories

sc = ax.scatter(
    df_tg['Mn_g_mol'], df_tg['Tg_K'],
    c=monomer_codes, cmap='tab10',
    alpha=0.65, s=50, edgecolors='white', linewidths=0.5
)

ax.set_xlabel('Number-average molecular weight Mₙ (g/mol)', fontsize=12)
ax.set_ylabel('Glass transition temperature Tg (K)', fontsize=12)
ax.set_title('Tg vs. Molecular Weight — Polymer Dataset\n(colour = monomer type)', fontsize=13)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.spines[['top', 'right']].set_visible(False)

# Legend
handles = [plt.Line2D([0],[0], marker='o', color='w',
    markerfacecolor=plt.cm.tab10(i/len(monomer_names)), markersize=8,
    label=name) for i, name in enumerate(monomer_names)]
ax.legend(handles=handles, title='Monomer type', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.savefig('tg_vs_mw_raw.png', dpi=150)
plt.show()

**C2 Observation** *(answer in this markdown cell before running C3)*

In 2 sentences:
1. What trend (if any) do you observe between Mₙ and Tg?
2. Do you notice any suspicious data points? Describe them.

*Your answer here:*

In [ ]:
# Cell C3 — Experimental design audit
# INSTRUCTOR DEMO

print("=== Experimental Design Audit ===")
print()

# Q1: How many unique polymer families?
n_families = df_tg['monomer_type'].nunique()
print(f"Q1. Unique polymer families: {n_families}")
print(f"    Families: {list(df_tg['monomer_type'].unique())}")
print()

# Q2: Is there replication?
# Replication = same polymer + same MW measured more than once
replication_check = df_tg.groupby(['monomer_type', 'Mn_g_mol']).size()
replicated = replication_check[replication_check > 1]
print(f"Q2. Replicated (monomer + MW) conditions: {len(replicated)}")
if len(replicated) > 0:
    print(f"    → Some conditions measured multiple times — check for conflicting Tg.")
print()

# Q3: What factors were varied?
print("Q3. Factors recorded in dataset:")
for col in df_tg.columns:
    n_unique = df_tg[col].nunique()
    print(f"    {col}: {n_unique} unique values")
print()

# Q4: Potential confounding variable
print("Q4. Potential confounding variables NOT recorded:")
print("    - Sample preparation conditions (solvent, casting method)")
print("    - DSC heating rate (affects apparent Tg significantly)")
print("    - Dispersity (Ð = Mw/Mn) — not recorded, but affects Tg")
print("    - Measurement lab / instrument (inter-lab variation)")

In [ ]:
# Cell C4 — quality_filter() function
# YOUR CODE HERE — complete the function

def quality_filter(df, tg_col='Tg_K', mw_col='Mn_g_mol',
                   id_col='polymer_id', tg_min=150.0):
    """
    Identify quality issues in a Tg polymer dataset.

    Returns
    -------
    report : dict with keys:
        'missing_mw'      : DataFrame rows with missing molecular weight
        'implausible_tg'  : DataFrame rows with Tg < tg_min (K)
        'conflicting_dups': DataFrame rows that are duplicates (same id)
                            with different Tg values
        'clean'           : DataFrame with all flagged rows removed
    """
    report = {}

    # 1. Missing molecular weight
    # YOUR CODE HERE
    report['missing_mw'] = None  # replace with your code

    # 2. Implausibly low Tg (likely unit error: °C entered as K)
    # YOUR CODE HERE
    report['implausible_tg'] = None  # replace with your code

    # 3. Duplicate entries with conflicting Tg values
    #    Hint: group by id_col, find groups where Tg values differ
    # YOUR CODE HERE
    report['conflicting_dups'] = None  # replace with your code

    # 4. Clean dataset: remove all flagged rows
    # YOUR CODE HERE
    report['clean'] = df.copy()  # replace with your filtered version

    return report


# Run the filter
report = quality_filter(df_tg)

print("Quality filter results:")
for key, val in report.items():
    if val is not None and hasattr(val, '__len__'):
        print(f"  {key}: {len(val)} rows")

In [ ]:
# Cell C5 — Before/after comparison
# After completing C4, run this to visualize the effect of your filter

df_clean = report['clean']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, df, title in [
    (ax1, df_tg,   f'Raw dataset (n={len(df_tg)})'),
    (ax2, df_clean, f'After quality filter (n={len(df_clean)})')
]:
    codes = df['monomer_type'].astype('category').cat.codes
    ax.scatter(df['Mn_g_mol'], df['Tg_K'], c=codes, cmap='tab10',
               alpha=0.65, s=40, edgecolors='white', linewidths=0.4)
    ax.set_xlabel('Mₙ (g/mol)', fontsize=11)
    ax.set_ylabel('Tg (K)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Effect of Quality Filter on Polymer Tg Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('tg_before_after_filter.png', dpi=150)
plt.show()

print(f"Removed {len(df_tg) - len(df_clean)} rows ({(len(df_tg)-len(df_clean))/len(df_tg)*100:.1f}%)")

---
## Part C — File Formats: CIF, POSCAR, XYZ
**In-class · Day 1 · Segment 4 + Lab (58–75 min)**

| Format | Contains | Used in | Parser |
|---|---|---|---|
| CIF | Lattice + species + symmetry | ICSD, literature | `Structure.from_file()` |
| POSCAR | Lattice + Cartesian/frac coords | VASP | `Structure.from_file()` |
| XYZ | Atomic positions only, no lattice | MD simulations | Limited — not for periodic solids |

In [ ]:
# Cell C6 — Read CIF, write POSCAR, read back
# INSTRUCTOR DEMO

from pymatgen.core import Structure

# Read the CIF
struct_cif = Structure.from_file('structures/BaTiO3.cif')
print("Read from CIF:")
print(f"  Formula:     {struct_cif.formula}")
print(f"  Space group: {struct_cif.get_space_group_info()}")
print(f"  Sites:       {len(struct_cif)}")
print()

# Write as POSCAR
struct_cif.to(filename='structures/POSCAR')
print("Written: structures/POSCAR")
print()

# Read the POSCAR back
struct_poscar = Structure.from_file('structures/POSCAR')
print("Read from POSCAR:")
print(f"  Formula:     {struct_poscar.formula}")
print(f"  Volume:      {struct_poscar.volume:.4f} Å³")
print()

# Verify round-trip
vol_match = abs(struct_cif.volume - struct_poscar.volume) < 1e-3
print(f"Round-trip volume match: {vol_match}")

In [ ]:
# Cell C7 — Task: read the provided TiO2 POSCAR and convert to CIF
# YOUR CODE HERE

# 1. Read 'structures/TiO2_POSCAR' using Structure.from_file()
# 2. Print: formula, space group, lattice parameters (a, b, c)
# 3. Write it as a CIF: 'structures/TiO2_rutile.cif'
# 4. Verify: read the CIF back and confirm volume matches

# YOUR CODE HERE


---
## Part D — Cross-Database Comparison *(Take-home)*

**Goal:** systematically compare bandgap values for 20 binary oxides across MP and JARVIS. This is the workflow you will use whenever you build an ML training set from multiple sources.

In [ ]:
# Cell D1 — Fetch 20 binary oxides from Materials Project
# Starter code — complete the JARVIS fetch and merge below

from mp_api.client import MPRester
from pymatgen.core import Composition

BINARY_OXIDE_IDS = [
    'mp-2657',   # TiO2 rutile
    'mp-19399',  # ZnO
    'mp-856',    # SnO2
    'mp-1143',   # MgO
    'mp-1265',   # Al2O3
    'mp-2172',   # WO3
    'mp-18904',  # Ga2O3
    'mp-390',    # Fe2O3 (hematite)
    'mp-19770',  # NiO
    'mp-1105',   # CoO
    'mp-1634',   # CuO
    'mp-1821',   # MnO
    'mp-2133',   # SrO
    'mp-2795',   # BaO
    'mp-1215',   # CaO
    'mp-612',    # Li2O
    'mp-2242',   # Na2O
    'mp-23703',  # Cr2O3
    'mp-510',    # V2O5
    'mp-504878', # In2O3
]

with MPRester(API_KEY) as mpr:
    mp_docs = mpr.materials.summary.search(
        material_ids=BINARY_OXIDE_IDS,
        fields=['material_id', 'formula_pretty', 'band_gap',
                'formation_energy_per_atom', 'energy_above_hull']
    )

df_mp = pd.DataFrame([{
    'mp_id':      d.material_id,
    'formula':    d.formula_pretty,
    'reduced':    Composition(d.formula_pretty).reduced_formula,
    'mp_gap':     d.band_gap,
    'mp_ef':      d.formation_energy_per_atom,
    'mp_ehull':   d.energy_above_hull,
} for d in mp_docs])

print(f"Fetched {len(df_mp)} MP entries")
print(df_mp[['mp_id','formula','mp_gap','mp_ef']].to_string(index=False))

In [ ]:
# Cell D2 — Build JARVIS lookup table for the same compounds
# YOUR CODE HERE

# The JARVIS data is already loaded in jd (from Cell B1).
# Build a DataFrame df_jarvis with columns:
#   'jid', 'formula', 'reduced', 'jarvis_gap', 'jarvis_ef'
#
# Hint: filter jd entries where v['formula'] matches any formula in df_mp['formula']
# Use Composition(formula).reduced_formula to normalise before matching
# For bandgap use 'optb88vdw_bandgap'; for formation energy use 'formation_energy_peratom'
# Some compounds may have multiple JARVIS entries (polymorphs) — keep the one with
# lowest formation energy.

# YOUR CODE HERE
df_jarvis = None  # replace with your DataFrame


In [ ]:
# Cell D3 — Merge on reduced formula
# YOUR CODE HERE

# Merge df_mp and df_jarvis on 'reduced'
# df_merged = pd.merge(df_mp, df_jarvis, on='reduced', how='inner')
# Print how many compounds matched

# YOUR CODE HERE
df_merged = None  # replace with your merged DataFrame


In [ ]:
# Cell D4 — Parity plot: MP bandgap vs. JARVIS bandgap
# YOUR CODE HERE

# Make a scatter plot with:
#   - x-axis: MP bandgap (GGA-PBE)
#   - y-axis: JARVIS bandgap (optB88vdW)
#   - dashed 1:1 reference line
#   - label each point with the reduced formula
#   - report in the title: MAE, R² of MP vs. JARVIS
# Identify and annotate the 3 largest outliers (biggest |Δgap|)

# YOUR CODE HERE


**D4 Analysis** *(answer in this markdown cell)*

1. What is the mean absolute difference between MP and JARVIS bandgaps for your 20 compounds?
2. Which 3 compounds show the largest discrepancy? Look up their space groups and crystal systems — is there a pattern?
3. In 3 sentences: what explains the discrepancy pattern you observe? (Think: DFT functional, treatment of d-electrons, van der Waals corrections.)
4. For your own research materials class, which database would you choose for an ML training set? Why?

*Your answer here:*

---
## Part E — Reflection *(Take-home)*

### E1 — Database choice for Tg prediction

You are building an ML training dataset for predicting glass transition temperature (Tg) of polymers.

In 2–3 sentences: which database would you use — a computed database like MP, an experimental dataset like the one in Part C, or both? Justify your choice in terms of **data quantity**, **data quality**, and **the scientific question** being addressed.

*Your answer here (≥ 2 sentences):*

### E2 — Redesigning the Tg experiment for ML

Redesign the Tg polymer experiment from Part C to produce a dataset better suited for ML training. Specify:

- What factors you would vary (e.g. monomer type, Mₙ, dispersity, architecture)
- How many levels per factor
- How many replicates per condition
- What confounding variables you would control (e.g. DSC heating rate, solvent history)
- Your estimate of the total number of samples the design would generate

*Your answer here:*

In [ ]:
# Cell E3 — Retrieve a material of your choice from JARVIS-DFT
# YOUR CODE HERE

# 1. Choose any material from JARVIS (pick something related to your research)
# 2. Print: formula, JID, at least one computed property
# 3. Find the same compound in MP and report the same property
# 4. Calculate the difference
# 5. In a markdown cell below: write 2–3 sentences explaining why they differ

MY_JID = 'JVASP-????'  # replace

# YOUR CODE HERE


**E3 Explanation** *(2–3 sentences)*

*Why does the property value differ between JARVIS and MP for your chosen material?*

*Your answer here:*

---
## Submission Checklist

Before submitting via LMS (due **Sunday 11:59 PM**):

**In-class (Parts A–C)**
- [ ] Cell A3: SrTiO₃ row added to DataFrame, reflection answered
- [ ] Cell A5: Ti fractional coordinates printed and interpreted
- [ ] Cell B4: your material fetched from both databases, comparison table printed
- [ ] Cell B4 Reflection: answered in markdown
- [ ] Cell C2 Observation: written in markdown
- [ ] Cell C4: `quality_filter()` complete — all three flag types working
- [ ] Cell C5: before/after plot generated
- [ ] Cell C7: TiO₂ POSCAR → CIF round-trip verified

**Take-home (Parts D–E)**
- [ ] Cell D2: JARVIS lookup table built
- [ ] Cell D3: DataFrames merged on reduced formula
- [ ] Cell D4: parity plot with MAE, R², outlier annotations
- [ ] D4 Analysis: answered in markdown
- [ ] E1: database choice justified in markdown
- [ ] E2: experiment redesign specified in markdown
- [ ] Cell E3: your JARVIS material fetched and compared
- [ ] E3 Explanation: answered in markdown

**Final steps**
- [ ] **Kernel → Restart & Run All** — confirm all cells execute without errors
- [ ] All output visible (plots displayed, DataFrames printed)
- [ ] Save notebook before uploading